## WP004 — Recalibration Layer (diagnostic)

WP003 showed the model over-shrinks: probabilities pulled toward the middle, under-confident on favourites, and its RPS gap to Pinnacle (**+0.0139**, CI [+0.0080, +0.0198]) *grows* with disagreement. This notebook asks: **is that gap just miscalibration, or are the model's match rankings genuinely worse than the market's?**

Approach: fit a cross-validated recalibration map on the model's own CV predictions (no retraining, no leakage), re-score on WP003's exact 401-match comparison, and see how much of the gap to Pinnacle each map closes. Temperature scaling is the primary test — one scalar that sharpens every probability. If it closes most of the gap, the model's ordering was fine and this is a cheap win. If it barely moves, the model genuinely ranks worse and structural work (priors / pooling) is needed — that becomes WP005.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar, minimize
from sklearn.isotonic import IsotonicRegression

from football_model.model.predict import dc_outcome_probs

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'
WP004 = REPO / 'work_products' / 'wp004_recalibration'

with open(WP001 / 'cv_checkpoint.pkl', 'rb') as f:
    cp = pickle.load(f)
with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv, windows, match_preds = shared['df_cv'], shared['windows'], cp['cv_match_predictions']
print(len(match_preds), 'model predictions across', len({m['window'] for m in match_preds}), 'windows')

### 1. Reconstruct fixtures + model probabilities (same as WP003)

In [ ]:
df_sorted = df_cv.sort_values('datetime').reset_index(drop=True)
first_round = df_cv.groupby('season')['round'].min().to_dict()
CODE_TO_FD = {
    'ARS': 'Arsenal', 'AVL': 'Aston Villa', 'BOU': 'Bournemouth', 'BRE': 'Brentford',
    'BRI': 'Brighton', 'BUR': 'Burnley', 'CHE': 'Chelsea', 'CRY': 'Crystal Palace',
    'EVE': 'Everton', 'FLH': 'Fulham', 'IPS': 'Ipswich', 'LED': 'Leeds',
    'LEI': 'Leicester', 'LIV': 'Liverpool', 'LUT': 'Luton', 'MCI': 'Man City',
    'MUN': 'Man United', 'NEW': 'Newcastle', 'NOR': 'Norwich', 'NOT': "Nott'm Forest",
    'SHE': 'Sheffield United', 'SOU': 'Southampton', 'SUN': 'Sunderland', 'TOT': 'Tottenham',
    'WAT': 'Watford', 'WBA': 'West Brom', 'WHU': 'West Ham', 'WOL': 'Wolves',
}
rows = []
for w in sorted({m['window'] for m in match_preds}):
    win = windows[w - 1]
    sel = df_sorted[(df_sorted['is_home'] == 1) & (df_sorted['round'] >= win['test_start']) & (df_sorted['round'] <= win['test_end'])]
    wp = [m for m in match_preds if m['window'] == w]
    assert len(sel) == len(wp)
    for (_, r), m in zip(sel.iterrows(), wp):
        assert int(r['goals_home']) == m['goals_home'] and int(r['goals_away']) == m['goals_away']
        rows.append({
            'window': w, 'date': pd.Timestamp(r['datetime']).normalize(), 'season': r['season'],
            'season_round': int(r['round']) - first_round[r['season']] + 1,
            'home_code': r['team'], 'away_code': r['opp_team'],
            'home_fd': CODE_TO_FD[r['team']], 'away_fd': CODE_TO_FD[r['opp_team']],
            'goals_home': m['goals_home'], 'goals_away': m['goals_away'],
            'lambda_home': m['lambda_home'], 'lambda_away': m['lambda_away'], 'rho_dc': m.get('rho_dc'),
        })
model_df = pd.DataFrame(rows)
P_raw = np.array([dc_outcome_probs(r.lambda_home, r.lambda_away, rho=r.rho_dc) for r in model_df.itertuples()])
model_df[['p_home_model', 'p_draw_model', 'p_away_model']] = P_raw
model_df['result'] = np.where(model_df['goals_home'] > model_df['goals_away'], 'H',
                       np.where(model_df['goals_home'] == model_df['goals_away'], 'D', 'A'))
# one-hot, column order H, D, A (matches p_home/p_draw/p_away)
Y = np.zeros((len(model_df), 3))
Y[np.arange(len(model_df)), model_df['result'].map({'H': 0, 'D': 1, 'A': 2}).values] = 1.0
print(len(model_df), 'fixtures reconstructed; class counts H/D/A =', Y.sum(0).astype(int))

### 2. Recalibration maps

All maps operate on the 3-vector `p = (p_H, p_D, p_A)` and are **cross-fitted leave-one-window-out**: to recalibrate window *w*, the map is fit on all matches *not* in window *w*. No fixture's recalibrated probability is influenced by its own outcome.

- **temperature** — `softmax(log(p) / T)`, one scalar `T` fit by minimising multiclass log-loss. `T<1` sharpens (the fix for over-shrinkage), `T>1` softens. The cleanest test of "is it just global under-confidence".
- **temperature (RPS-fit)** — same, but `T` chosen to minimise mean RPS instead of log-loss. Sensitivity check — does optimising the decision-relevant loss change the answer.
- **vector Platt** — per-class 2-parameter logistic on `logit(p_c)`, then renormalise. More capacity: can fix per-outcome bias (e.g. draws) as well as global sharpness.
- **isotonic** — per-class monotonic non-parametric fit of `1[outcome=c]` on `p_c`, then renormalise. Most capacity; risk of overfitting the ~390-match calibration set.

In [ ]:
EPS = 1e-6

def _clip(P):
    P = np.clip(P, EPS, 1 - EPS)
    return P / P.sum(axis=1, keepdims=True)

def logloss(P, Y):
    return -(Y * np.log(_clip(P))).sum(axis=1).mean()

def rps_vec(P, Y):
    # ordered H,D,A: cumulative over the ordering
    cp = np.cumsum(P[:, :2], axis=1)   # [P(H), P(H)+P(D)]
    cy = np.cumsum(Y[:, :2], axis=1)
    return 0.5 * ((cp - cy) ** 2).sum(axis=1).mean()

# ---- temperature ----
def apply_temperature(P, T):
    z = np.log(_clip(P))
    zz = z / T
    zz -= zz.max(axis=1, keepdims=True)
    e = np.exp(zz)
    return e / e.sum(axis=1, keepdims=True)

def fit_temperature(P, Y, loss='logloss'):
    fn = logloss if loss == 'logloss' else rps_vec
    r = minimize_scalar(lambda logT: fn(apply_temperature(P, np.exp(logT)), Y),
                        bounds=(np.log(0.2), np.log(5.0)), method='bounded')
    return float(np.exp(r.x))

# ---- vector Platt (per-class logistic on logit(p_c)) ----
def _logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def fit_platt(P, Y):
    params = []
    for c in range(3):
        x, y = _logit(P[:, c]), Y[:, c]
        def nll(ab, x=x, y=y):
            a, b = ab
            z = a * x + b
            # stable binary cross-entropy
            return np.mean(np.logaddexp(0, z) - y * z)
        r = minimize(nll, x0=[1.0, 0.0], method='Nelder-Mead')
        params.append(tuple(r.x))
    return params

def apply_platt(P, params):
    Q = np.empty_like(P)
    for c, (a, b) in enumerate(params):
        z = a * _logit(P[:, c]) + b
        Q[:, c] = 1 / (1 + np.exp(-z))
    return Q / Q.sum(axis=1, keepdims=True)

# ---- isotonic (per-class) ----
def fit_isotonic(P, Y):
    models = []
    for c in range(3):
        ir = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
        ir.fit(P[:, c], Y[:, c])
        models.append(ir)
    return models

def apply_isotonic(P, models):
    Q = np.column_stack([models[c].predict(P[:, c]) for c in range(3)])
    Q = np.clip(Q, EPS, None)
    return Q / Q.sum(axis=1, keepdims=True)

# ---- leave-one-window-out cross-fit ----
def crossfit(P, Y, wins, fit_fn, apply_fn, **fit_kw):
    out = np.empty_like(P)
    fitted = {}
    for w in np.unique(wins):
        tr = wins != w
        params = fit_fn(P[tr], Y[tr], **fit_kw)
        out[wins == w] = apply_fn(P[wins == w], params)
        fitted[int(w)] = params
    return out, fitted

wins = model_df['window'].values
P_temp, T_folds = crossfit(P_raw, Y, wins, fit_temperature, apply_temperature)
P_temp_rps, Trps_folds = crossfit(P_raw, Y, wins, fit_temperature, apply_temperature, loss='rps')
P_platt, _ = crossfit(P_raw, Y, wins, fit_platt, apply_platt)
P_iso, _ = crossfit(P_raw, Y, wins, fit_isotonic, apply_isotonic)

Tv = np.array(list(T_folds.values()))
print(f'temperature T across folds (log-loss fit): mean {Tv.mean():.3f}  sd {Tv.std():.3f}  range [{Tv.min():.3f}, {Tv.max():.3f}]')
Tr = np.array(list(Trps_folds.values()))
print(f'temperature T across folds (RPS fit):      mean {Tr.mean():.3f}  sd {Tr.std():.3f}')
print(f'T on all 401 (ship value, log-loss): {fit_temperature(P_raw, Y):.3f}')
for name, P in [('raw', P_raw), ('temp', P_temp), ('temp-RPS', P_temp_rps), ('platt', P_platt), ('isotonic', P_iso)]:
    print(f'{name:>9}  crossfit logloss {logloss(P, Y):.4f}   RPS {rps_vec(P, Y):.4f}')

### 3. Bring in the odds (from WP003's cache) and join

In [ ]:
odds_raw = pd.read_pickle(WP003 / 'odds_raw.pkl')
BOOK_COLS = {
    'pinnacle': [('PSCH', 'PSCD', 'PSCA'), ('PSH', 'PSD', 'PSA')],
    'b365':     [('B365CH', 'B365CD', 'B365CA'), ('B365H', 'B365D', 'B365A')],
    'avg':      [('AvgCH', 'AvgCD', 'AvgCA'), ('AvgH', 'AvgD', 'AvgA')],
    'max':      [('MaxCH', 'MaxCD', 'MaxCA'), ('MaxH', 'MaxD', 'MaxA')],
}
BOOK_PICK = {}
for book, opts in BOOK_COLS.items():
    for cols in opts:
        if all(c in odds_raw.columns for c in cols):
            BOOK_PICK[book] = cols
            break

# attach recalibrated probs to model_df before the join
for name, P in [('temp', P_temp), ('tempRPS', P_temp_rps), ('platt', P_platt), ('iso', P_iso)]:
    model_df[[f'p_home_{name}', f'p_draw_{name}', f'p_away_{name}']] = P

j = model_df.merge(odds_raw, left_on=['date', 'home_fd', 'away_fd'],
                   right_on=['Date', 'HomeTeam', 'AwayTeam'], how='left', indicator=True)
matched = j[j['_merge'] == 'both'].copy()
assert (matched['goals_home'] == matched['FTHG']).all() and (matched['goals_away'] == matched['FTAG']).all()
print(f'{len(matched)}/{len(j)} joined')

def devig_prop(o):
    inv = 1.0 / np.asarray(o, float)
    return inv / inv.sum()

for book, cols in BOOK_PICK.items():
    ok = matched[list(cols)].notna().all(axis=1)
    Pm = np.full((len(matched), 3), np.nan)
    Pm[ok.values] = np.array([devig_prop(r) for r in matched.loc[ok, list(cols)].to_numpy()])
    matched[f'p_home_{book}'], matched[f'p_draw_{book}'], matched[f'p_away_{book}'] = Pm[:, 0], Pm[:, 1], Pm[:, 2]
    print(f'{book}: {int(ok.sum())} rows')

### 4. Pooled RPS — raw vs each recalibration vs market

In [ ]:
def rps_row(ph, pd_, pa, actual):
    cp1, cp2 = ph, ph + pd_
    ce1 = 1.0 if actual == 'H' else 0.0
    ce2 = 1.0 if actual in ('H', 'D') else 0.0
    return 0.5 * ((cp1 - ce1) ** 2 + (cp2 - ce2) ** 2)

def rps_series(dfm, name):
    return dfm.apply(lambda x: rps_row(x[f'p_home_{name}'], x[f'p_draw_{name}'], x[f'p_away_{name}'], x['result']), axis=1)

def boot_ci(v, n_boot=5000, seed=0):
    v = np.asarray(v, float); rng = np.random.default_rng(seed)
    bm = np.array([rng.choice(v, len(v), replace=True).mean() for _ in range(n_boot)])
    return v.mean(), *np.percentile(bm, [2.5, 97.5])

avg_h, avg_a = df_cv['goals_home'].mean(), df_cv['goals_away'].mean()
naive_probs = dc_outcome_probs(avg_h, avg_a)

rows = []
for name, label in [('model', 'model (raw)'), ('temp', 'model + temperature'), ('tempRPS', 'model + temperature (RPS-fit)'),
                    ('platt', 'model + vector Platt'), ('iso', 'model + isotonic')]:
    m, lo, hi = boot_ci(rps_series(matched, name).values)
    rows.append({'predictor': label, 'n': len(matched), 'RPS': round(m, 4), 'CI': f'[{lo:.4f}, {hi:.4f}]'})
for b in [x for x in ['pinnacle', 'b365', 'avg'] if f'p_home_{x}' in matched.columns]:
    s = matched.dropna(subset=[f'p_home_{b}'])
    m, lo, hi = boot_ci(rps_series(s, b).values)
    rows.append({'predictor': b, 'n': len(s), 'RPS': round(m, 4), 'CI': f'[{lo:.4f}, {hi:.4f}]'})
nr = matched['result'].map(lambda a: rps_row(*naive_probs, a))
m, lo, hi = boot_ci(nr.values)
rows.append({'predictor': 'naive', 'n': len(nr), 'RPS': round(m, 4), 'CI': f'[{lo:.4f}, {hi:.4f}]'})
print(pd.DataFrame(rows).to_string(index=False))

### 5. How much of the gap to Pinnacle does each map close?

Paired bootstrap of `RPS(candidate) - RPS(Pinnacle)` on the Pinnacle subset. Raw gap was **+0.0139**; `% closed = 1 - new_gap / 0.0139`.

In [ ]:
s = matched.dropna(subset=['p_home_pinnacle']).copy()
rp = rps_series(s, 'pinnacle').values
raw_gap = (rps_series(s, 'model').values - rp).mean()
print(f'n = {len(s)}   raw model - Pinnacle gap = {raw_gap:+.4f}\n')
print(f'{"candidate":>28}   {"gap vs Pinnacle":>16}   {"95% CI":>22}   {"% of raw gap closed":>19}')
for name, label in [('model', 'raw'), ('temp', 'temperature'), ('tempRPS', 'temperature (RPS-fit)'),
                    ('platt', 'vector Platt'), ('iso', 'isotonic')]:
    d = rps_series(s, name).values - rp
    m, lo, hi = boot_ci(d)
    closed = 1 - m / raw_gap
    print(f'{label:>28}   {m:>+16.4f}   [{lo:+.4f}, {hi:+.4f}]   {closed:>18.0%}')

### 6. Calibration — raw vs best map vs Pinnacle (P(home win))

In [ ]:
def reliability(p, y, n_bins=8):
    p, y = np.asarray(p, float), np.asarray(y, float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        msk = idx == b
        if msk.sum():
            out.append({'bin': f'{edges[b]:.2f}-{edges[b+1]:.2f}', 'n': int(msk.sum()),
                        'pred': round(p[msk].mean(), 3), 'actual': round(y[msk].mean(), 3)})
    return pd.DataFrame(out)

yh = (matched['result'] == 'H').astype(int).values
for name, label in [('model', 'RAW MODEL'), ('temp', 'MODEL + TEMPERATURE'), ('platt', 'MODEL + VECTOR PLATT')]:
    print(f'{label} — P(home win):')
    print(reliability(matched[f'p_home_{name}'].values, yh).to_string(index=False)); print()
sp = matched.dropna(subset=['p_home_pinnacle'])
print('PINNACLE — P(home win):')
print(reliability(sp['p_home_pinnacle'].values, (sp['result'] == 'H').astype(int).values).to_string(index=False))

### 7. Does recalibration fix the "more wrong where it disagrees" pattern?

WP003: raw model's RPS gap to Pinnacle *grew* with disagreement (+0.014 all → +0.041 top-10%). Re-run for the best map.

In [ ]:
mkt = 'pinnacle'
d0 = matched.dropna(subset=[f'p_home_{mkt}']).copy()
d0['rps_mkt'] = rps_series(d0, mkt).values
for name, label in [('model', 'RAW'), ('temp', 'TEMPERATURE'), ('platt', 'VECTOR PLATT')]:
    d = d0.copy()
    d['rps_c'] = rps_series(d, name).values
    d['disag'] = (d[f'p_home_{name}'] - d[f'p_home_{mkt}']).abs()
    print(f'--- {label} ---')
    for lab, q in [('all', 0.0), ('top 50%', 0.5), ('top 25%', 0.75), ('top 10%', 0.90)]:
        sub = d[d['disag'] >= d['disag'].quantile(q)]
        g = sub['rps_c'].mean() - sub['rps_mkt'].mean()
        _, lo, hi = boot_ci((sub['rps_c'] - sub['rps_mkt']).values)
        print(f'  {lab:<8} n={len(sub):>3}  gap {g:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]')
    print()

### 8. Betting sim — raw vs best map

Same as WP003 §10: flat stake where `P_candidate` beats the `Max` implied prob by τ, settled at `Max`.

In [ ]:
if 'max' in BOOK_PICK:
    mc = BOOK_PICK['max']
    sim = matched.dropna(subset=list(mc)).copy()
    for name, label in [('model', 'RAW'), ('temp', 'TEMPERATURE'), ('platt', 'VECTOR PLATT')]:
        legs = []
        for _, x in sim.iterrows():
            for out_, pc, oc in [('H', f'p_home_{name}', mc[0]), ('D', f'p_draw_{name}', mc[1]), ('A', f'p_away_{name}', mc[2])]:
                legs.append({'edge': x[pc] - 1.0 / x[oc], 'odds': x[oc], 'won': x['result'] == out_})
        L = pd.DataFrame(legs)
        print(f'--- {label} ---   {"tau":>5} {"n":>5} {"roi":>9}   95% CI')
        for tau in [0.0, 0.05, 0.10]:
            b = L[L['edge'] >= tau]
            if len(b) == 0:
                print(f'{"":>17}{tau:>5.2f} {0:>5}'); continue
            pr = np.where(b['won'], b['odds'] - 1.0, -1.0)
            _, lo, hi = boot_ci(pr)
            print(f'{"":>17}{tau:>5.2f} {len(b):>5} {pr.mean():>+9.1%}   [{lo:+.1%}, {hi:+.1%}]')
        print()
else:
    print('no Max columns')

### 9. Verdict + ship value

If temperature scaling closes most of the gap and flattens the disagreement pattern: the model's rankings were fine, ship the single `T` below in the prediction path (apply `softmax(log(p)/T)` to `dc_outcome_probs` output). If it barely moves: escalate to WP005 (structural — loosen `sigma_att/def`, `home_adv_sd`, AR `rho`), measured on this same comparison.

In [ ]:
T_ship = fit_temperature(P_raw, Y)
T_ship_rps = fit_temperature(P_raw, Y, loss='rps')
print(f'Ship value  T = {T_ship:.4f}   (RPS-fit alternative: {T_ship_rps:.4f})')
print(f'T < 1 => sharpen (model was under-confident); T > 1 => soften')
# quick summary line
g_raw = raw_gap
g_temp = (rps_series(s, "temp").values - rp).mean()
print(f'\nGap to Pinnacle:  raw {g_raw:+.4f}  ->  +temperature {g_temp:+.4f}   ({1 - g_temp / g_raw:.0%} closed)')